In [1]:
import numpy as np
import pandas as pd

In [2]:
df_train = pd.read_csv("/kaggle/input/datasets/kennethrodrigues/operon-pair-classification-v2/operon_train.csv")
df_val = pd.read_csv("/kaggle/input/datasets/kennethrodrigues/operon-pair-classification-v2/operon_val.csv")

df_train.head()

,sequence_1,sequence_2,operon_id,label
0,MQKQHNVVAGVDMGATHIRFCLRTAEGETLHCEKKRTAEVIAPGLV...,MSQSEFDSALPNGIGLAPYLRMKQEGMTENESRIVEWLLKPGNLSC...,KO03198,1
1,MFRGATLVNLDSKGRLSVPTRYREQLLENAAGQMVCTIDIYHPCLL...,MKAAAKTQKPKRQEEHANFISWRFALLCGCILLALAFLLGRVAWLQ...,KO03290,1
2,MEQFSAFKLLLKKQYETTLGFYDKYIKNLKRFALKNNVLFVIVDNE...,MAISKKKRFFFDLAQDEDDAETVQEVKKVEQQLKLEPVVQPQHDLT...,KO07485,1
3,MTTQYGFFIDSSRCTGCKTCELACKDFKDLGPEVSFRRIYEYAGGD...,MGNGWHEWPLVIFTVLGQCVVGALIVSGIGWFAAKNDADRQRIVRG...,KO03811,1
4,MTSPNNYLAKIKVVGVGGGGVNAVNRMIEEGLKGVEFIAVNTDSQA...,MTTGASKKPARPNTGAKTRTGLGIRERISGAWNDLLARPLTDYIMI...,KO13645,1


In [3]:
from collections import Counter
from typing import Tuple
 
 
# ---------------------------------------------------------------------------
# Lookup tables
# ---------------------------------------------------------------------------
 
STANDARD_AAS = list("ACDEFGHIKLMNPQRSTVWY")
 
# Kyte-Doolittle hydrophobicity scale
HYDROPHOBICITY = {
    "A":  1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C":  2.5,
    "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2, "I":  4.5,
    "L":  3.8, "K": -3.9, "M":  1.9, "F":  2.8, "P": -1.6,
    "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V":  4.2,
}
 
# Approximate residue molecular weights (Da)
RESIDUE_MW = {
    "A":  89.1, "R": 174.2, "N": 132.1, "D": 133.1, "C": 121.2,
    "Q": 146.2, "E": 147.1, "G":  75.0, "H": 155.2, "I": 131.2,
    "L": 131.2, "K": 146.2, "M": 149.2, "F": 165.2, "P": 115.1,
    "S": 105.1, "T": 119.1, "W": 204.2, "Y": 181.2, "V": 117.1,
}
 
# Net charge at pH 7 (simplified: +1 for K/R, -1 for D/E, +0.1 for H)
CHARGE_AT_PH7 = {
    "K": +1.0, "R": +1.0, "H": +0.1,
    "D": -1.0, "E": -1.0,
}
 
# Amino acid groups (5 groups → used for group frequencies + group bigrams)
AA_GROUPS = {
    "hydrophobic":         set("AVILMFYW"),      # non-polar/aliphatic + aromatic
    "polar_uncharged":     set("STNQ"),
    "positively_charged":  set("KRH"),
    "negatively_charged":  set("DE"),
    "special":             set("CGP"),            # Cys (disulfide), Gly (flexible), Pro (rigid)
}
GROUP_NAMES = list(AA_GROUPS.keys())             # fixed order for reproducibility
GROUP_OF = {}
for grp, members in AA_GROUPS.items():
    for aa in members:
        GROUP_OF[aa] = grp
 
 
# ---------------------------------------------------------------------------
# Feature block 1: Amino acid frequencies  (20 features)
# ---------------------------------------------------------------------------
 
def _aa_frequencies(seq: str) -> np.ndarray:
    """
    Fraction of each of the 20 standard amino acids.
    Unknown residues are ignored in the denominator contribution.
    """
    counts = Counter(seq)
    total = sum(counts.get(aa, 0) for aa in STANDARD_AAS)
    if total == 0:
        return np.zeros(20)
    return np.array([counts.get(aa, 0) / total for aa in STANDARD_AAS])
 
 
# ---------------------------------------------------------------------------
# Feature block 2: Amino acid group frequencies  (5 features)
# ---------------------------------------------------------------------------
 
def _group_frequencies(seq: str) -> np.ndarray:
    """
    Fraction of residues belonging to each of the 5 biochemical groups.
    """
    counts = {g: 0 for g in GROUP_NAMES}
    valid = 0
    for aa in seq:
        g = GROUP_OF.get(aa)
        if g:
            counts[g] += 1
            valid += 1
    if valid == 0:
        return np.zeros(5)
    return np.array([counts[g] / valid for g in GROUP_NAMES])
 
 
# ---------------------------------------------------------------------------
# Feature block 3: Physicochemical aggregate statistics  (7 features)
# ---------------------------------------------------------------------------
 
def _physicochemical(seq: str) -> np.ndarray:
    """
    Seven scalar summaries derived from per-residue physicochemical scales.
 
    Index  Feature
    -----  -------
    0      Mean hydrophobicity  (Kyte-Doolittle)
    1      Std  hydrophobicity
    2      Net charge at pH 7
    3      Approximate molecular weight (kDa)
    4      Aromaticity  (fraction F + W + Y)
    5      Aliphatic index proxy  (fraction A + V + I + L)
    6      Isoelectric-point proxy  (charge / sqrt(length), clamped)
    """
    hydro  = [HYDROPHOBICITY.get(aa, 0.0) for aa in seq]
    charge = sum(CHARGE_AT_PH7.get(aa, 0.0) for aa in seq)
    mw     = sum(RESIDUE_MW.get(aa, 110.0) for aa in seq) / 1000.0   # → kDa
    n      = len(seq) if seq else 1
 
    aromaticity = sum(1 for aa in seq if aa in "FWY") / n
    aliphatic   = sum(1 for aa in seq if aa in "AVIL") / n
    pi_proxy    = np.clip(charge / np.sqrt(n), -5, 5)
 
    return np.array([
        np.mean(hydro) if hydro else 0.0,
        np.std(hydro)  if hydro else 0.0,
        charge,
        mw,
        aromaticity,
        aliphatic,
        pi_proxy,
    ])
 
 
# ---------------------------------------------------------------------------
# Feature block 4: Sequence complexity / structural  (4 features)
# ---------------------------------------------------------------------------
 
def _complexity(seq: str) -> np.ndarray:
    """
    Four features capturing sequence-level complexity.
 
    Index  Feature
    -----  -------
    0      Sequence length (log-scaled so extreme lengths don't dominate)
    1      Shannon entropy over amino acid frequencies
    2      Max run length (longest unbroken repeat of a single residue) / length
    3      Fraction of unique amino acids present (richness)
    """
    n = len(seq)
    if n == 0:
        return np.zeros(4)
 
    # Shannon entropy
    freqs = np.array(list(Counter(seq).values())) / n
    entropy = float(-np.sum(freqs * np.log2(freqs + 1e-12)))
 
    # Max run length
    max_run = 1
    cur_run = 1
    for i in range(1, n):
        if seq[i] == seq[i - 1]:
            cur_run += 1
            max_run = max(max_run, cur_run)
        else:
            cur_run = 1
 
    unique_fraction = len(set(seq) & set(STANDARD_AAS)) / 20.0
 
    return np.array([
        np.log1p(n),              # log(1 + length)
        entropy,
        max_run / n,
        unique_fraction,
    ])
 
 
# ---------------------------------------------------------------------------
# Feature block 5: Group-level bigrams  (25 features)
# ---------------------------------------------------------------------------
 
def _group_bigrams(seq: str) -> np.ndarray:
    """
    Transition frequencies between the 5 biochemical groups.
    Produces a flattened 5×5 matrix (25 features) where entry (i, j)
    is the fraction of consecutive pairs (group_i → group_j).
 
    This captures local sequence order at the group level, avoiding
    the sparsity of full 20×20 amino-acid bigrams (400 features).
    """
    matrix = np.zeros((5, 5))
    g_idx  = {g: i for i, g in enumerate(GROUP_NAMES)}
    total  = 0
 
    for k in range(len(seq) - 1):
        gi = GROUP_OF.get(seq[k])
        gj = GROUP_OF.get(seq[k + 1])
        if gi and gj:
            matrix[g_idx[gi], g_idx[gj]] += 1
            total += 1
 
    if total > 0:
        matrix /= total
 
    return matrix.flatten()   # 25 features
 
 
# ---------------------------------------------------------------------------
# Core per-sequence feature extractor  (61 features total)
# ---------------------------------------------------------------------------
 
def extract_sequence_features(seq: str) -> np.ndarray:
    """
    Convert a single amino acid sequence into a 61-dimensional feature vector.
 
    Block breakdown:
        [0:20]   AA frequencies          (20 features)
        [20:25]  AA group frequencies    ( 5 features)
        [25:32]  Physicochemical stats   ( 7 features)
        [32:36]  Complexity / structure  ( 4 features)
        [36:61]  Group-level bigrams     (25 features)
                                          ----------
                                          61 total
    """
    seq = seq.upper().strip()
    return np.concatenate([
        _aa_frequencies(seq),    # 20
        _group_frequencies(seq), #  5
        _physicochemical(seq),   #  7
        _complexity(seq),        #  4
        _group_bigrams(seq),     # 25
    ])
 
 
# ---------------------------------------------------------------------------
# Public API: pair → input matrix row  (305 features)
# ---------------------------------------------------------------------------
 
def aa_pair_to_feature_vector(seq_a: str, seq_b: str) -> np.ndarray:
    """
    Convert a pair of amino acid sequences into a single 305-dimensional
    feature vector suitable for scikit-learn's LogisticRegression.
 
    The vector is assembled using the Siamese interaction pattern:
 
        [u | v | u−v | |u−v| | u⊙v]
 
    where u = extract_sequence_features(seq_a)
          v = extract_sequence_features(seq_b)
 
    Parameters
    ----------
    seq_a : str
        First amino acid sequence (single-letter codes, any length).
    seq_b : str
        Second amino acid sequence (single-letter codes, any length).
 
    Returns
    -------
    np.ndarray, shape (305,)
        Feature vector ready for logistic regression.
    """
    u = extract_sequence_features(seq_a)
    v = extract_sequence_features(seq_b)
 
    return np.concatenate([
        u,              # individual representation of seq_a
        v,              # individual representation of seq_b
        u - v,          # signed difference  → directional divergence
        np.abs(u - v),  # absolute difference → magnitude of divergence
        u * v,          # element-wise product → shared signal
    ])
 
 
def aa_pairs_to_matrix(pairs: list) -> np.ndarray:
    """
    Convert a list of (seq_a, seq_b) tuples into a 2-D design matrix
    of shape (n_samples, 305).
 
    Parameters
    ----------
    pairs : list of (str, str)
        Each element is a tuple of two amino acid sequences.
 
    Returns
    -------
    np.ndarray, shape (n_samples, 305)
 
    Example
    -------
    >>> pairs = [
    ...     ("MKTAYIAKQR", "MKALLFDSSFR"),
    ...     ("ACDEFGHIKL", "ACDEFGHIKL"),
    ... ]
    >>> X = aa_pairs_to_matrix(pairs)
    >>> X.shape
    (2, 305)
    """
    return np.array([aa_pair_to_feature_vector(a, b) for a, b in pairs])
 
 
def get_feature_names() -> list:
    """
    Return an ordered list of 305 human-readable feature names
    matching the columns of aa_pairs_to_matrix().
    """
    names = []
 
    # --- Block 1: AA frequencies (u, v, diff, abs_diff, product)
    for prefix in ["u", "v", "diff", "absdiff", "prod"]:
        for block, features in [
            ("aa_freq",    [f"freq_{aa}"     for aa in STANDARD_AAS]),
            ("grp_freq",   [f"grp_{g}"       for g in GROUP_NAMES]),
            ("physchem",   ["hydro_mean", "hydro_std", "net_charge",
                            "mol_weight_kDa", "aromaticity",
                            "aliphatic", "pi_proxy"]),
            ("complexity", ["log_length", "entropy", "max_run_frac",
                            "unique_aa_frac"]),
            ("bigrams",    [f"bg_{GROUP_NAMES[i][:3]}_{GROUP_NAMES[j][:3]}"
                            for i in range(5) for j in range(5)]),
        ]:
            if prefix == "u":
                names.extend([f"u__{block}__{f}" for f in features])
 
    # rebuild in the correct [u | v | diff | absdiff | prod] order
    raw_names = []
    per_seq_names = []
    for block, features in [
        ("aa_freq",    [f"freq_{aa}"     for aa in STANDARD_AAS]),
        ("grp_freq",   [f"grp_{g}"       for g in GROUP_NAMES]),
        ("physchem",   ["hydro_mean", "hydro_std", "net_charge",
                        "mol_weight_kDa", "aromaticity",
                        "aliphatic", "pi_proxy"]),
        ("complexity", ["log_length", "entropy", "max_run_frac",
                        "unique_aa_frac"]),
        ("bigrams",    [f"bg_{GROUP_NAMES[i][:3]}_{GROUP_NAMES[j][:3]}"
                        for i in range(5) for j in range(5)]),
    ]:
        per_seq_names.extend([f"{block}__{f}" for f in features])
 
    result = []
    for prefix in ["u", "v", "diff", "absdiff", "prod"]:
        result.extend([f"{prefix}__{n}" for n in per_seq_names])
    return result

In [4]:
def validate(pipeline, val_pairs: list, val_labels: list,
             class_names: Tuple[str, str] = ("different operon", "same operon")) -> dict:
    """
    Evaluate a fitted pipeline on held-out validation pairs and print a
    structured report.
 
    Parameters
    ----------
    pipeline : fitted sklearn Pipeline
        Must already have been fit on training data.
    val_pairs : list of (str, str)
        Held-out sequence pairs — never seen during training.
    val_labels : list of int
        Ground-truth labels matching val_pairs (0 = class_names[0], 1 = class_names[1]).
    class_names : tuple of str
        Human-readable names for class 0 and class 1.
 
    Returns
    -------
    dict with keys: accuracy, precision, recall, f1, roc_auc, confusion_matrix
    """
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score,
        f1_score, roc_auc_score, confusion_matrix, classification_report,
    )
 
    X_val = aa_pairs_to_matrix(val_pairs)
    y_val = np.array(val_labels)
 
    y_pred      = pipeline.predict(X_val)
    y_prob      = pipeline.predict_proba(X_val)[:, 1]   # P(same operon)
 
    acc       = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall    = recall_score(y_val, y_pred, zero_division=0)
    f1        = f1_score(y_val, y_pred, zero_division=0)
    roc_auc   = roc_auc_score(y_val, y_prob) if len(np.unique(y_val)) > 1 else float("nan")
    cm        = confusion_matrix(y_val, y_pred)
 
    # ── Pretty print ──────────────────────────────────────────────────────
    sep = "─" * 52
    print(f"\n{'VALIDATION REPORT':^52}")
    print(sep)
    print(f"  Samples evaluated : {len(y_val)}")
    print(f"  Class 0           : {class_names[0]}  (n={int((y_val==0).sum())})")
    print(f"  Class 1           : {class_names[1]}  (n={int((y_val==1).sum())})")
    print(sep)
    print(f"  Accuracy          : {acc:.4f}")
    print(f"  Precision         : {precision:.4f}   ← of predicted positives, % correct")
    print(f"  Recall            : {recall:.4f}   ← of true positives, % found")
    print(f"  F1 Score          : {f1:.4f}   ← harmonic mean of P and R")
    print(f"  ROC-AUC           : {roc_auc:.4f}   ← 0.5 = random, 1.0 = perfect")
    print(sep)
    print("  Confusion matrix:")
    print(f"                    Pred 0   Pred 1")
    print(f"    Actual 0  :    {cm[0,0]:>5}    {cm[0,1]:>5}   (TN, FP)")
    print(f"    Actual 1  :    {cm[1,0]:>5}    {cm[1,1]:>5}   (FN, TP)")
    print(sep)
    print("\n  Per-class breakdown:")
    print(classification_report(y_val, y_pred,
                                target_names=list(class_names),
                                zero_division=0))
 
    return dict(accuracy=acc, precision=precision, recall=recall,
                f1=f1, roc_auc=roc_auc, confusion_matrix=cm)
 

In [5]:

# Example: operon pair classification
# Label 1 = same operon, 0 = different operon
pairs = []
labels = []

for idx, row in df_train.iterrows():
    pairs.append((row['sequence_1'], row['sequence_2']))
    labels.append(row['label'])



In [6]:
print(len(pairs), len(labels))

37126 37126


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


In [8]:
X = aa_pairs_to_matrix(pairs)
y = np.array(labels)

print(f"Feature matrix shape: {X.shape}")   # (4, 305)
print(f"Feature names (first 5): {get_feature_names()[:5]}")

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(C=1.0, max_iter=1000,
                                  class_weight="balanced")),
])
pipeline.fit(X, y)
print(f"Training accuracy: {pipeline.score(X, y):.2f}")
 

Feature matrix shape: (37126, 305)
Feature names (first 5): ['u__aa_freq__freq_A', 'u__aa_freq__freq_C', 'u__aa_freq__freq_D', 'u__aa_freq__freq_E', 'u__aa_freq__freq_F']
Training accuracy: 0.63


In [9]:
import joblib
joblib.dump(pipeline, 'logistic_pipline.joblib')

['logistic_pipline.joblib']

In [10]:
val_pairs = []
val_labels = []
for idx, row in df_val.iterrows():
    val_pairs.append((row['sequence_1'], row['sequence_2']))
    val_labels.append(row['label'])


In [11]:
metrics = validate(pipeline, val_pairs, val_labels)


                 VALIDATION REPORT                  
────────────────────────────────────────────────────
  Samples evaluated : 15912
  Class 0           : different operon  (n=7966)
  Class 1           : same operon  (n=7946)
────────────────────────────────────────────────────
  Accuracy          : 0.6247
  Precision         : 0.6249   ← of predicted positives, % correct
  Recall            : 0.6219   ← of true positives, % found
  F1 Score          : 0.6234   ← harmonic mean of P and R
  ROC-AUC           : 0.6713   ← 0.5 = random, 1.0 = perfect
────────────────────────────────────────────────────
  Confusion matrix:
                    Pred 0   Pred 1
    Actual 0  :     4999     2967   (TN, FP)
    Actual 1  :     3004     4942   (FN, TP)
────────────────────────────────────────────────────

  Per-class breakdown:
                  precision    recall  f1-score   support

different operon       0.62      0.63      0.63      7966
     same operon       0.62      0.62      0.62    

In [12]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

def train_and_validate_xgb(
    train_pairs, train_labels,
    val_pairs,   val_labels,
):
    X_train = aa_pairs_to_matrix(train_pairs)
    X_val   = aa_pairs_to_matrix(val_pairs)
    y_train = np.array(train_labels)
    y_val   = np.array(val_labels)

    pos   = y_train.sum()
    neg   = len(y_train) - pos

    clf = XGBClassifier(
        n_estimators      = 300,
        max_depth         = 6,
        learning_rate     = 0.05,
        scale_pos_weight  = neg / pos,   # handles class imbalance
        eval_metric       = "auc",
        early_stopping_rounds = 20,      # stops if val AUC stops improving
        random_state      = 42,
    )

    clf.fit(
        X_train, y_train,
        eval_set      = [(X_val, y_val)],
        verbose       = False,
    )

    y_pred = clf.predict(X_val)
    y_prob = clf.predict_proba(X_val)[:, 1]

    print(classification_report(y_val, y_pred,
          target_names=["different operon", "same operon"]))
    print(f"ROC-AUC: {roc_auc_score(y_val, y_prob):.4f}")
    print(f"Best iteration: {clf.best_iteration}")

    return clf

In [13]:
clf = train_and_validate_xgb(pairs, labels, val_pairs, val_labels)

                  precision    recall  f1-score   support

different operon       0.73      0.74      0.73      7966
     same operon       0.74      0.72      0.73      7946

        accuracy                           0.73     15912
       macro avg       0.73      0.73      0.73     15912
    weighted avg       0.73      0.73      0.73     15912

ROC-AUC: 0.8135
Best iteration: 299


In [14]:
joblib.dump(clf, 'xgboost_clf.joblib')

['xgboost_clf.joblib']

In [15]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

def test_xgb(
    val_pairs,   val_labels, clf
):
    X_val   = aa_pairs_to_matrix(val_pairs)
    y_val   = np.array(val_labels)

    y_pred = clf.predict(X_val)
    y_prob = clf.predict_proba(X_val)[:, 1]

    print(classification_report(y_val, y_pred,
          target_names=["different operon", "same operon"]))
    print(f"ROC-AUC: {roc_auc_score(y_val, y_prob):.4f}")
    print(f"Best iteration: {clf.best_iteration}")

In [16]:
cyano = pd.read_parquet("hf://datasets/tattabio/cyano_operonic_pair/data/train-00000-of-00001.parquet")
vibrio = pd.read_parquet("hf://datasets/tattabio/vibrio_operonic_pair/data/train-00000-of-00001.parquet")
ecoli = pd.read_parquet("hf://datasets/tattabio/ecoli_operonic_pair/data/train-00000-of-00001.parquet")

In [17]:
tatta_bio_df = pd.concat([cyano, vibrio, ecoli],ignore_index=True)

In [18]:
def create_pairs(df: pd.DataFrame) -> pd.DataFrame:
    # Extract genome accession and positional index from entry
    df = df.copy()
    df['genome'] = df['Entry'].str.extract(r'lcl\|(.*?)_prot')
    df['idx'] = df['Entry'].str.extract(r'_(\d+)$').astype(int)
    
    # Sort by genome and position
    df = df.sort_values(['genome', 'idx']).reset_index(drop=True)
    
    pairs = []
    for genome, group in df.groupby('genome'):
        group = group.reset_index(drop=True)
        for i in range(len(group) - 1):
            # Only pair if indices are consecutive
            if group.loc[i+1, 'idx'] == group.loc[i, 'idx'] + 1:
                pairs.append({
                    'sequence_1': group.loc[i, 'Sequence'],
                    'sequence_2': group.loc[i+1, 'Sequence'],
                    'label': group.loc[i, 'Label']
                })
    
    return pd.DataFrame(pairs)

In [19]:
tatta_pairs = create_pairs(tatta_bio_df)
tatta_pairs.head()

,sequence_1,sequence_2,label
0,MLWQDCDQRLGQPPPMKLVCRQNELNTSLSLVSRAVPSRPNHPVLA...,MAMSDQPEELRVSDLLNRQVLDRQTTDEHGRIDRVWMHPPAHRVLG...,1
1,MAMSDQPEELRVSDLLNRQVLDRQTTDEHGRIDRVWMHPPAHRVLG...,MTAISSAPFSADEIAGEGIKPEEYDAIVERLGRHPNKAELGMFGVM...,0
2,MTAISSAPFSADEIAGEGIKPEEYDAIVERLGRHPNKAELGMFGVM...,MIPTQPLTADLDCDLGLERPDRPEEACGVFALYAPGEEVARMAYFG...,1
3,MIPTQPLTADLDCDLGLERPDRPEEACGVFALYAPGEEVARMAYFG...,MLNRDRPRWFVGQDLDGFLGLALDNLIQILLILGLCQGLLGYPPSL...,0
4,MLNRDRPRWFVGQDLDGFLGLALDNLIQILLILGLCQGLLGYPPSL...,MTPIFWLGISFLLVASSLSVLLWVAVPAVQELARTARSAQRLLDML...,0


In [20]:
cyano_pairs = []
cyano_labels = []

for idx, row in tatta_pairs.iterrows():
    cyano_pairs.append((row['sequence_1'], row['sequence_2']))
    cyano_labels.append(row['label'])

In [21]:
import joblib
pipeline = joblib.load('logistic_pipline.joblib')

In [22]:
test_metrics = validate(pipeline, cyano_pairs, cyano_labels)


                 VALIDATION REPORT                  
────────────────────────────────────────────────────
  Samples evaluated : 5183
  Class 0           : different operon  (n=3556)
  Class 1           : same operon  (n=1627)
────────────────────────────────────────────────────
  Accuracy          : 0.6076
  Precision         : 0.4064   ← of predicted positives, % correct
  Recall            : 0.5433   ← of true positives, % found
  F1 Score          : 0.4650   ← harmonic mean of P and R
  ROC-AUC           : 0.6252   ← 0.5 = random, 1.0 = perfect
────────────────────────────────────────────────────
  Confusion matrix:
                    Pred 0   Pred 1
    Actual 0  :     2265     1291   (TN, FP)
    Actual 1  :      743      884   (FN, TP)
────────────────────────────────────────────────────

  Per-class breakdown:
                  precision    recall  f1-score   support

different operon       0.75      0.64      0.69      3556
     same operon       0.41      0.54      0.47     

In [23]:
clf = joblib.load('xgboost_clf.joblib')

In [24]:
test_xgb(cyano_pairs, cyano_labels, clf)

                  precision    recall  f1-score   support

different operon       0.74      0.65      0.70      3556
     same operon       0.40      0.51      0.45      1627

        accuracy                           0.61      5183
       macro avg       0.57      0.58      0.57      5183
    weighted avg       0.64      0.61      0.62      5183

ROC-AUC: 0.6160
Best iteration: 299
